# Chapter 1: Introduction to Data
**Haydar Kilic — Artificial Intelligence Engineering**

---
This notebook demonstrates the theoretical concepts from the Chapter 1 lecture slides through hands-on Python examples.

**Contents:**
1. Case Study: Chronic Fatigue Syndrome
2. Data Matrix and Variable Types
3. Relationships Between Variables
4. Sampling Bias
5. Sampling Strategies
6. Random Assignment and Inference

In [ ]:
# Required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

print('Libraries loaded successfully.')

---
## 1. Case Study: Chronic Fatigue Syndrome

> **Source:** Deale et al. (1997). *Cognitive behavior therapy for chronic fatigue syndrome: A randomized controlled trial.* The American Journal of Psychiatry, 154(3).

**Study Design:**
- 142 patients referred, 60 included (82 excluded)
- Treatment group (n=30): Cognitive behavioral therapy
- Control group (n=30): Relaxation techniques
- At 6-month follow-up, 7 patients dropped out (3 treatment, 4 control)

In [ ]:
# --- Create study data ---
outcomes = pd.DataFrame({
    'Group'          : ['Treatment', 'Treatment', 'Control', 'Control'],
    'Outcome'        : ['Good', 'Not Good', 'Good', 'Not Good'],
    'Patient Count'  : [19, 8, 5, 21]
})

# Summary table (contingency table)
pivot = outcomes.pivot(index='Group', columns='Outcome', values='Patient Count')
pivot['Total'] = pivot.sum(axis=1)
pivot.loc['Total'] = pivot.sum()

print('=== 6-Month Follow-Up Results ===')
print(pivot.to_string())

# Calculate proportions
prop_treatment = 19 / 27
prop_control   = 5  / 26
diff           = prop_treatment - prop_control

print(f'\nProportion with good outcome — Treatment : {prop_treatment:.2f}  ({prop_treatment*100:.0f}%)')
print(f'Proportion with good outcome — Control   : {prop_control:.2f}  ({prop_control*100:.0f}%)')
print(f'Difference between groups                : {diff:.2f}  ({diff*100:.0f}%)')

In [ ]:
# --- Visualize results ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = {'Good': '#2196F3', 'Not Good': '#BDBDBD'}

# Left: Stacked bar chart
ax = axes[0]
groups    = ['Treatment\n(n=27)', 'Control\n(n=26)']
good      = [19/27*100, 5/26*100]
not_good  = [8/27*100, 21/26*100]

bars1 = ax.bar(groups, good,     color='#2196F3', label='Good Outcome', width=0.5)
bars2 = ax.bar(groups, not_good, bottom=good, color='#BDBDBD', label='Not Good', width=0.5)

for bar, val in zip(bars1, good):
    ax.text(bar.get_x() + bar.get_width()/2, val/2,
            f'{val:.0f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=13)
for bar, val, bot in zip(bars2, not_good, good):
    ax.text(bar.get_x() + bar.get_width()/2, bot + val/2,
            f'{val:.0f}%', ha='center', va='center', color='#333', fontweight='bold', fontsize=13)

ax.set_title('Good Outcome Rate (6 Months)', fontsize=13, fontweight='bold')
ax.set_ylabel('Percentage (%)')
ax.set_ylim(0, 110)
ax.legend(loc='upper right')

# Right: Patient flow diagram
ax2 = axes[1]
ax2.axis('off')
ax2.set_title('Patient Flow', fontsize=13, fontweight='bold')

box_style      = dict(boxstyle='round,pad=0.5', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=1.5)
excluded_style = dict(boxstyle='round,pad=0.5', facecolor='#FFF9C4', edgecolor='#F9A825', linewidth=1.5)
treatment_style= dict(boxstyle='round,pad=0.5', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=1.5)
control_style  = dict(boxstyle='round,pad=0.5', facecolor='#FCE4EC', edgecolor='#880E4F', linewidth=1.5)

ax2.text(0.5, 0.92, '142 Patients Referred', ha='center', va='center',
         transform=ax2.transAxes, bbox=box_style, fontsize=10)
ax2.annotate('', xy=(0.5, 0.78), xytext=(0.5, 0.84),
             xycoords='axes fraction', arrowprops=dict(arrowstyle='->', color='#555'))
ax2.text(0.5, 0.73, '60 Patients Enrolled', ha='center', va='center',
         transform=ax2.transAxes, bbox=box_style, fontsize=10)
ax2.text(0.82, 0.86, '82 Excluded\n(diagnosis/refusal/other illness)', ha='center', va='center',
         transform=ax2.transAxes, bbox=excluded_style, fontsize=8.5)
ax2.annotate('', xy=(0.5, 0.59), xytext=(0.5, 0.65),
             xycoords='axes fraction', arrowprops=dict(arrowstyle='->', color='#555'))
ax2.text(0.22, 0.53, 'Treatment Group\n(n=30)', ha='center', va='center',
         transform=ax2.transAxes, bbox=treatment_style, fontsize=10)
ax2.text(0.78, 0.53, 'Control Group\n(n=30)', ha='center', va='center',
         transform=ax2.transAxes, bbox=control_style, fontsize=10)
ax2.text(0.5, 0.59, 'Random Assignment', ha='center', va='center',
         transform=ax2.transAxes, fontsize=8.5, color='#555', style='italic')
ax2.text(0.22, 0.35, '3 Dropped Out\n→ 27 remaining', ha='center', va='center',
         transform=ax2.transAxes, fontsize=9, color='#C62828')
ax2.text(0.78, 0.35, '4 Dropped Out\n→ 26 remaining', ha='center', va='center',
         transform=ax2.transAxes, fontsize=9, color='#C62828')
ax2.text(0.22, 0.18, 'Good Outcome: 19/27\n(70%)', ha='center', va='center',
         transform=ax2.transAxes, bbox=treatment_style, fontsize=10, fontweight='bold')
ax2.text(0.78, 0.18, 'Good Outcome: 5/26\n(19%)', ha='center', va='center',
         transform=ax2.transAxes, bbox=control_style, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.suptitle('Chronic Fatigue Syndrome RCT Results', fontsize=14,
             fontweight='bold', y=1.01)
plt.show()

In [ ]:
# --- Chi-square test: Is the difference statistically significant? ---
from scipy.stats import chi2_contingency, fisher_exact

# Observed contingency table
observed = np.array([[19, 8],   # Treatment: good, not good
                     [5,  21]]) # Control:   good, not good

chi2, p_chi2, dof, expected = chi2_contingency(observed, correction=False)
odds_ratio, p_fisher = fisher_exact(observed)

print('=== Statistical Significance Tests ===')
print(f'\nChi-square test:')
print(f'  χ² = {chi2:.3f},  df = {dof},  p = {p_chi2:.5f}')
print(f"\nFisher's exact test (more appropriate for small samples):")
print(f'  Odds ratio = {odds_ratio:.2f},  p = {p_fisher:.5f}')

alpha = 0.05
if p_chi2 < alpha:
    print(f'\n→ p < {alpha}: The difference is statistically SIGNIFICANT.')
    print('  The null hypothesis that the treatment effect is due to chance is rejected.')
else:
    print(f'\n→ p ≥ {alpha}: The difference is NOT statistically significant.')

---
## 2. Data Matrix and Variable Types

Let us model the survey data from the slides as a pandas DataFrame and classify each variable by type.

| Variable | Type |
|---|---|
| `gender` | Categorical |
| `sleep` | Numerical, Continuous |
| `bedtime` | Categorical, Ordinal |
| `countries` | Numerical, Discrete |
| `fear` | Categorical, Ordinal (can also be treated as numerical) |

In [ ]:
# --- Create survey data ---
np.random.seed(42)
n = 86  # Number of students in the slides

gender_options  = ['male', 'female']
ie_options      = ['introvert', 'extrovert']
time_options    = ['22:00-00:00', '00:00-02:00', '02:00+']

df = pd.DataFrame({
    'gender'            : np.random.choice(gender_options, n, p=[0.45, 0.55]),
    'introvert_extrovert': np.random.choice(ie_options, n, p=[0.4, 0.6]),
    'sleep'             : np.round(np.random.normal(6.5, 1.2, n).clip(3, 10), 1),
    'bedtime'           : np.random.choice(time_options, n, p=[0.35, 0.50, 0.15]),
    'countries'         : np.random.randint(0, 20, n),
    'fear'              : np.random.randint(1, 6, n)
})

# Define ordinal variable as pandas Categorical
time_order = pd.CategoricalDtype(
    categories=['22:00-00:00', '00:00-02:00', '02:00+'], ordered=True)
df['bedtime'] = df['bedtime'].astype(time_order)

print('First 6 Rows of the Data Matrix:')
print(df.head(6).to_string(index=True))
print(f'\nDimensions: {df.shape[0]} observations × {df.shape[1]} variables')

In [ ]:
# --- Inspect variable types ---
print('=== Variable Types ===')
print(f"{'Variable':<25} {'Pandas dtype':<20} {'Statistical Type'}")
print('-' * 70)
info = [
    ('gender',              'object',   'Categorical (nominal)'),
    ('introvert_extrovert', 'object',   'Categorical (nominal)'),
    ('sleep',               'float64',  'Numerical, Continuous'),
    ('bedtime',             'category', 'Categorical, Ordinal'),
    ('countries',           'int64',    'Numerical, Discrete'),
    ('fear',                'int64',    'Categorical, Ordinal / Numerical'),
]
for variable, dtype, stat_type in info:
    print(f"{variable:<25} {dtype:<20} {stat_type}")

In [ ]:
# --- Variable types: Taxonomy diagram ---
fig, ax = plt.subplots(figsize=(11, 5))
ax.axis('off')
ax.set_title('Variable Type Taxonomy', fontsize=14, fontweight='bold', pad=15)

def box(ax, x, y, text, width=0.14, height=0.10,
        color='#E3F2FD', edge='#1565C0', font=9.5):
    box_obj = mpatches.FancyBboxPatch(
        (x - width/2, y - height/2), width, height,
        boxstyle='round,pad=0.01', facecolor=color, edgecolor=edge, linewidth=1.5,
        transform=ax.transAxes)
    ax.add_patch(box_obj)
    ax.text(x, y, text, ha='center', va='center',
            transform=ax.transAxes, fontsize=font, fontweight='bold')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                xycoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Root
box(ax, 0.50, 0.88, 'All Variables', width=0.20, color='#1565C0', edge='#0D47A1',
     font=10.5)
ax.texts[-1].set_color('white')

# First level
box(ax, 0.27, 0.62, 'Numerical', color='#E8F5E9', edge='#2E7D32')
box(ax, 0.73, 0.62, 'Categorical', color='#FFF3E0', edge='#E65100')

arrow(ax, 0.42, 0.84, 0.30, 0.68)
arrow(ax, 0.58, 0.84, 0.70, 0.68)

# Second level
box(ax, 0.14, 0.35, 'Continuous\n\nex: sleep, height', color='#F1F8E9', edge='#558B2F')
box(ax, 0.40, 0.35, 'Discrete\n\nex: country count', color='#F1F8E9', edge='#558B2F')
box(ax, 0.62, 0.35, 'Nominal\n\nex: gender', color='#FFF8E1', edge='#F9A825')
box(ax, 0.87, 0.35, 'Ordinal\n\nex: bedtime,\nfear', color='#FFF8E1', edge='#F9A825')

arrow(ax, 0.22, 0.58, 0.17, 0.41)
arrow(ax, 0.30, 0.58, 0.37, 0.41)
arrow(ax, 0.68, 0.58, 0.65, 0.41)
arrow(ax, 0.78, 0.58, 0.84, 0.41)

plt.tight_layout()
plt.show()

In [ ]:
# --- Summary statistics for each variable ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Distributions of Survey Variables', fontsize=14, fontweight='bold')

# gender
df['gender'].value_counts().plot(kind='bar', ax=axes[0,0],
    color=['#42A5F5','#EF5350'], edgecolor='white', width=0.5)
axes[0,0].set_title('Gender (Categorical)', fontweight='bold')
axes[0,0].set_xlabel('')
axes[0,0].tick_params(axis='x', rotation=0)

# introvert/extrovert
df['introvert_extrovert'].value_counts().plot(kind='bar', ax=axes[0,1],
    color=['#66BB6A','#FFA726'], edgecolor='white', width=0.5)
axes[0,1].set_title('Introvert / Extrovert (Categorical)', fontweight='bold')
axes[0,1].set_xlabel('')
axes[0,1].tick_params(axis='x', rotation=0)

# sleep — continuous numerical
axes[0,2].hist(df['sleep'], bins=12, color='#26A69A', edgecolor='white')
axes[0,2].axvline(df['sleep'].mean(), color='#C62828', linestyle='--',
                  label=f"Mean: {df['sleep'].mean():.1f}")
axes[0,2].set_title('Sleep (Numerical, Continuous)', fontweight='bold')
axes[0,2].set_xlabel('Hours')
axes[0,2].legend()

# bedtime — ordinal categorical
times = df['bedtime'].value_counts().sort_index()
times.plot(kind='bar', ax=axes[1,0],
    color=['#7986CB','#9575CD','#BA68C8'], edgecolor='white', width=0.5)
axes[1,0].set_title('Bedtime (Ordinal Categorical)', fontweight='bold')
axes[1,0].set_xlabel('')
axes[1,0].tick_params(axis='x', rotation=15)

# countries — discrete numerical
axes[1,1].hist(df['countries'], bins=15, color='#FF7043', edgecolor='white')
axes[1,1].set_title('Countries Visited (Numerical, Discrete)', fontweight='bold')
axes[1,1].set_xlabel('Number of Countries')

# fear — ordinal categorical / numerical
df['fear'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1,2],
    color=['#EF9A9A','#EF5350','#E53935','#C62828','#B71C1C'],
    edgecolor='white', width=0.5)
axes[1,2].set_title('Fear (Ordinal, 1–5)', fontweight='bold')
axes[1,2].set_xlabel('Fear Level')
axes[1,2].tick_params(axis='x', rotation=0)

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## 3. Relationships Between Variables

**Key Concepts:**
- **Explanatory (independent) variable** → **Response (dependent) variable**
- If there is an association between two variables → **Correlated**
- **Association ≠ Causation** (causation can only be inferred from a randomized experiment)

Let us simulate the GPA–study hours example from the slides.

In [ ]:
# --- GPA - Study hours relationship ---
np.random.seed(7)
n_students = 120

study_hours = np.random.exponential(scale=15, size=n_students).clip(1, 70)
noise       = np.random.normal(0, 0.25, n_students)
gpa         = (3.0 + 0.012 * study_hours + noise).clip(2.8, 4.0)

# Outlier from the slides: student with GPA > 4.0
study_hours = np.append(study_hours, 22)
gpa         = np.append(gpa, 4.15)   # data error / outlier

df_gpa = pd.DataFrame({'study_hours': study_hours, 'gpa': gpa})

# Correlation
r, p_val = stats.pearsonr(df_gpa['study_hours'], df_gpa['gpa'])

fig, ax = plt.subplots(figsize=(9, 5))

# Normal points
normal_mask = df_gpa['gpa'] <= 4.0
ax.scatter(df_gpa.loc[normal_mask, 'study_hours'],
           df_gpa.loc[normal_mask, 'gpa'],
           color='#64B5F6', alpha=0.7, s=50, label='Normal observation')

# Outlier
ax.scatter(df_gpa.loc[~normal_mask, 'study_hours'],
           df_gpa.loc[~normal_mask, 'gpa'],
           color='#E53935', s=100, zorder=5,
           label='Outlier (GPA > 4.0 → data error?)')
ax.annotate('GPA > 4.0?\n(data error)',
            xy=(22, 4.15), xytext=(35, 4.13),
            arrowprops=dict(arrowstyle='->', color='#E53935'),
            color='#E53935', fontsize=9)

# Regression line (excluding outlier)
m, b = np.polyfit(df_gpa.loc[normal_mask, 'study_hours'],
                  df_gpa.loc[normal_mask, 'gpa'], 1)
x_line = np.linspace(1, 70, 100)
ax.plot(x_line, m*x_line + b, color='#1565C0', linewidth=2,
        linestyle='--', label=f'Regression line (r = {r:.2f})')

ax.axhline(4.0, color='#F9A825', linestyle=':', linewidth=1.5, label='GPA = 4.0 ceiling')
ax.set_xlabel('Weekly Study Hours', fontsize=11)
ax.set_ylabel('GPA', fontsize=11)
ax.set_title('Relationship Between GPA and Weekly Study Hours', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 75)

print(f'Pearson correlation coefficient: r = {r:.3f}, p = {p_val:.4f}')
print('Interpretation: Weak-to-moderate positive association; however, this does NOT prove causation.')
plt.tight_layout()
plt.show()

In [ ]:
# --- Possum: Head length - Skull width ---
# Practice example from the slides
try:
    import statsmodels.api as sm  # for dataset if available
except ImportError:
    pass

np.random.seed(21)
n_possum = 104
head_length  = np.random.normal(92, 3.5, n_possum).clip(84, 103)
skull_width  = 0.55 * head_length + np.random.normal(0, 1.8, n_possum)

r_p, _ = stats.pearsonr(head_length, skull_width)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(head_length, skull_width,
           color='#5C6BC0', alpha=0.65, s=45)
m2, b2 = np.polyfit(head_length, skull_width, 1)
x2 = np.linspace(84, 103, 100)
ax.plot(x2, m2*x2+b2, 'r--', linewidth=1.8, label=f'r = {r_p:.2f}')
ax.set_xlabel('Head Length (mm)', fontsize=11)
ax.set_ylabel('Skull Width (mm)', fontsize=11)
ax.set_title('Possum: Head Length – Skull Width', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

# Answer box
ax.text(0.03, 0.95,
        'Correct answer (c): Positive association\nCausation cannot be inferred!',
        transform=ax.transAxes, fontsize=9,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='#FFF9C4', edgecolor='#F9A825'))

plt.tight_layout()
plt.show()

---
## 4. Sampling Bias

Three main types of sampling bias are discussed in the slides:

| Type | Description |
|---|---|
| **Non-response bias** | The majority of sampled individuals do not participate in the survey |
| **Voluntary response bias** | Only those with strong opinions respond |
| **Convenience sampling bias** | Individuals who are easiest to access are selected |

### Historical Example: Literary Digest 1936 Poll

In [ ]:
# --- Literary Digest 1936: Biased sample simulation ---
np.random.seed(99)

# True US electorate (1936)
# FDR: 62%, Landon: 38%
N_population = 100_000
population = np.random.choice(['FDR', 'Landon'], size=N_population, p=[0.62, 0.38])

# Literary Digest sample: wealthy segment (telephone/car owners)
# This group tended to favor Republicans
N_digest = 2_400_000  # approximate number of responses (simulated proportionally)
biased_sample = np.random.choice(['FDR', 'Landon'], size=10_000, p=[0.43, 0.57])

# True random sample
random_sample = np.random.choice(population, size=10_000)

def calc_proportion(array, candidate):
    return (array == candidate).mean() * 100

print('=== Literary Digest 1936 Simulation ===')
print(f"\nActual election result       : FDR 62% – Landon 38%")
print(f"Misleading poll estimate (Digest): FDR {calc_proportion(biased_sample,'FDR'):.0f}% – "
      f"Landon {calc_proportion(biased_sample,'Landon'):.0f}%")
print(f"Random sample estimate           : FDR {calc_proportion(random_sample,'FDR'):.0f}% – "
      f"Landon {calc_proportion(random_sample,'Landon'):.0f}%")

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Biased Sampling: 1936 US Presidential Election', fontsize=13, fontweight='bold')

data = [
    ('Actual Result', [62, 38]),
    ('Literary Digest\n(Biased Sample)', [43, 57]),
    ('Random Sample', [calc_proportion(random_sample,'FDR'),
                       calc_proportion(random_sample,'Landon')]),
]

bar_colors = ['#1565C0', '#E53935', '#2E7D32']
for ax, (title, proportions), color in zip(axes, data, bar_colors):
    candidates = ['FDR', 'Landon']
    bar_color  = [color, '#BDBDBD']
    bars = ax.bar(candidates, proportions, color=bar_color, width=0.5, edgecolor='white')
    for bar, val in zip(bars, proportions):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.0f}%', ha='center', fontweight='bold', fontsize=13)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylim(0, 80)
    ax.set_ylabel('Vote Share (%)')
    ax.axhline(50, color='#555', linestyle=':', linewidth=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\nConclusion: A biased survey of 2.4 million respondents produced worse predictions")
print("       than a small but well-constructed random sample. A large biased sample cannot compensate for bias!")

---
## 5. Sampling Strategies

Let us simulate three fundamental sampling methods in Python:
1. **Simple Random Sampling (SRS)**
2. **Stratified Sampling**
3. **Cluster Sampling**

In [ ]:
# --- Sampling Strategies: Visualization ---
np.random.seed(2025)
N = 300          # population size
n_sample = 30    # sample size

# Create population: 3 strata / 6 clusters
x = np.random.uniform(0, 10, N)
y = np.random.uniform(0, 6, N)

# Stratum assignment (3 groups based on y-axis)
stratum = pd.cut(y, bins=[0, 2, 4, 6], labels=['Stratum 1', 'Stratum 2', 'Stratum 3'])

# Cluster assignment (grid)
cluster_x = pd.cut(x, bins=3, labels=False)
cluster_y = pd.cut(y, bins=2, labels=False)
cluster   = cluster_x * 2 + cluster_y  # 6 clusters indexed 0..5

df_population = pd.DataFrame({'x': x, 'y': y, 'stratum': stratum, 'cluster': cluster})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Sampling Strategies', fontsize=14, fontweight='bold')

stratum_colors = {'Stratum 1': '#90CAF9', 'Stratum 2': '#A5D6A7', 'Stratum 3': '#FFCC80'}
cluster_colors = plt.cm.Set2(np.linspace(0, 1, 6))

# 1) Simple Random Sampling
ax = axes[0]
selected_idx = np.random.choice(N, n_sample, replace=False)
ax.scatter(x, y, color='#90CAF9', alpha=0.4, s=25, label='Population')
ax.scatter(x[selected_idx], y[selected_idx],
           color='#E53935', s=80, zorder=5, label='Selected')
ax.set_title('Simple Random Sampling', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=9)
ax.text(0.02, 0.97, f'n = {n_sample}', transform=ax.transAxes,
        va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))

# 2) Stratified Sampling
ax = axes[1]
selected_strat = []
for strat, grp in df_population.groupby('stratum', observed=True):
    n_strat = max(1, round(n_sample * len(grp) / N))
    selected_strat.extend(grp.sample(n_strat, random_state=42).index.tolist())

for strat, color in stratum_colors.items():
    mask = df_population['stratum'] == strat
    ax.scatter(df_population.loc[mask, 'x'], df_population.loc[mask, 'y'],
               color=color, alpha=0.45, s=25, label=strat)
    sel_mask = mask & df_population.index.isin(selected_strat)
    ax.scatter(df_population.loc[sel_mask, 'x'], df_population.loc[sel_mask, 'y'],
               color='#B71C1C', s=80, zorder=5)

# Show stratum boundaries
for yval in [2, 4]:
    ax.axhline(yval, color='#555', linestyle='--', linewidth=1, alpha=0.7)
ax.set_title('Stratified Sampling', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=8, loc='upper right')

# 3) Cluster Sampling
ax = axes[2]
selected_clusters = np.random.choice(6, 3, replace=False)  # select 3 clusters

for k in range(6):
    mask = df_population['cluster'] == k
    if k in selected_clusters:
        ax.scatter(df_population.loc[mask, 'x'], df_population.loc[mask, 'y'],
                   color=cluster_colors[k], s=60, zorder=5,
                   edgecolors='#B71C1C', linewidths=1.2,
                   label=f'Cluster {k+1} (selected)')
    else:
        ax.scatter(df_population.loc[mask, 'x'], df_population.loc[mask, 'y'],
                   color=cluster_colors[k], alpha=0.3, s=25)

ax.set_title('Cluster Sampling', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=8, loc='upper right')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# --- Stratified sampling with pandas ---
# Proportional stratified sampling by gender on the student data
np.random.seed(42)

target_sample = 30  # total sample size

def stratified_sample(df, stratum_col, n_total, random_state=42):
    """Performs proportional stratified sampling."""
    sample_parts = []
    for stratum, group in df.groupby(stratum_col):
        n_stratum = round(n_total * len(group) / len(df))
        n_stratum = max(1, min(n_stratum, len(group)))  # boundary check
        part = group.sample(n=n_stratum, random_state=random_state)
        sample_parts.append(part)
        print(f"  {stratum}: population={len(group)}, sample={n_stratum}")
    return pd.concat(sample_parts)

print('=== Stratified Sampling by Gender ===')
sample_df = stratified_sample(df, 'gender', target_sample)

print(f"\nPopulation gender distribution: {df['gender'].value_counts().to_dict()}")
print(f"Sample gender distribution     : {sample_df['gender'].value_counts().to_dict()}")
print(f"\nPopulation proportion (female) : {df['gender'].value_counts(normalize=True)['female']*100:.1f}%")
print(f"Sample proportion (female)     : {sample_df['gender'].value_counts(normalize=True)['female']*100:.1f}%")

---
## 6. Random Assignment and Inference

The 2×2 matrix from the slides:

| | Random Assignment | No Random Assignment |
|---|---|---|
| **Random Sampling** | Causation → generalizable to population (**ideal experiment**) | Correlation → generalizable to population |
| **No Random Sampling** | Causation → sample only | Correlation → sample only (**poor observational**) |

In [ ]:
# --- Importance of random assignment: Confounding variable simulation ---
# Example: Shoe size and reading ability
# True cause: AGE — it affects both variables (confounding variable)

np.random.seed(55)
n = 200

# Age (5-18), drives everything
age = np.random.randint(5, 19, n)

# Shoe size depends on age
shoe_size = 25 + 0.9 * age + np.random.normal(0, 1.5, n)

# Reading ability depends on age
reading = 10 + 5 * age + np.random.normal(0, 8, n)

r_raw, _ = stats.pearsonr(shoe_size, reading)

# Controlling for age (partial correlation)
res_shoe    = stats.linregress(age, shoe_size)
res_reading = stats.linregress(age, reading)
resid_shoe    = shoe_size - (res_shoe.slope    * age + res_shoe.intercept)
resid_reading = reading   - (res_reading.slope * age + res_reading.intercept)
r_partial, _ = stats.pearsonr(resid_shoe, resid_reading)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Confounding Variable Example',
             fontsize=13, fontweight='bold')

sc = axes[0].scatter(shoe_size, reading, c=age, cmap='RdYlBu_r', alpha=0.7, s=45)
plt.colorbar(sc, ax=axes[0], label='Age')
axes[0].set_xlabel('Shoe Size')
axes[0].set_ylabel('Reading Ability Score')
axes[0].set_title(f'Raw Association: r = {r_raw:.2f}\n(MISLEADING — confounder: age)',
                  fontsize=10, fontweight='bold')

axes[1].scatter(resid_shoe, resid_reading, color='#78909C', alpha=0.6, s=45)
m_r, b_r = np.polyfit(resid_shoe, resid_reading, 1)
x_r = np.linspace(resid_shoe.min(), resid_shoe.max(), 100)
axes[1].plot(x_r, m_r*x_r+b_r, 'r--', linewidth=2)
axes[1].set_xlabel('Shoe Size (age effect removed)')
axes[1].set_ylabel('Reading Ability (age effect removed)')
axes[1].set_title(f'After Controlling for Age: r = {r_partial:.2f}\n(No real association!)',
                  fontsize=10, fontweight='bold')
axes[1].axhline(0, color='#555', linewidth=0.8, linestyle=':')
axes[1].axvline(0, color='#555', linewidth=0.8, linestyle=':')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Conclusion:')
print(f'  Raw correlation (misleading)        : r = {r_raw:.2f}')
print(f'  After controlling for age (true)    : r = {r_partial:.2f}')
print('  → Association does not imply causation. The confounding variable (AGE) was driving both variables up together.')

In [ ]:
# --- Summary: Random Assignment × Random Sampling Matrix ---
fig, ax = plt.subplots(figsize=(11, 6))
ax.axis('off')
ax.set_title('Effect of Random Assignment and Sampling on Inference',
             fontsize=14, fontweight='bold', pad=20)

def box2(ax, x, y, text, sub_text, color, edge, width=0.30, height=0.25):
    box_obj = mpatches.FancyBboxPatch(
        (x - width/2, y - height/2), width, height,
        boxstyle='round,pad=0.015', facecolor=color, edgecolor=edge,
        linewidth=2, transform=ax.transAxes)
    ax.add_patch(box_obj)
    ax.text(x, y + 0.04, text, ha='center', va='center',
            transform=ax.transAxes, fontsize=10, fontweight='bold')
    ax.text(x, y - 0.06, sub_text, ha='center', va='center',
            transform=ax.transAxes, fontsize=8.5, style='italic', color='#333')

# Header row / column
ax.text(0.38, 0.90, 'Random Assignment YES', ha='center', va='center',
        transform=ax.transAxes, fontsize=11, fontweight='bold', color='#1565C0')
ax.text(0.72, 0.90, 'Random Assignment NO', ha='center', va='center',
        transform=ax.transAxes, fontsize=11, fontweight='bold', color='#B71C1C')
ax.text(0.05, 0.68, 'Random\nSampling\nYES', ha='center', va='center',
        transform=ax.transAxes, fontsize=10, fontweight='bold')
ax.text(0.05, 0.30, 'Random\nSampling\nNO', ha='center', va='center',
        transform=ax.transAxes, fontsize=10, fontweight='bold')

box2(ax, 0.38, 0.65, '✅ Ideal Experiment',
     'Causal conclusion\nGeneralizable to population',
     '#E8F5E9', '#2E7D32')
box2(ax, 0.72, 0.65, '⚠️ Observational Study\n(Good)',
     'Correlation conclusion\nGeneralizable to population',
     '#FFF9C4', '#F9A825')
box2(ax, 0.38, 0.28, '🔬 Most Experiments',
     'Causal conclusion\nSample only',
     '#E3F2FD', '#1565C0')
box2(ax, 0.72, 0.28, '❌ Poor Observational',
     'Correlation conclusion\nSample only',
     '#FFEBEE', '#C62828')

# Axis labels
ax.text(0.55, 0.97, '← Causation →                      ← Correlation →',
        ha='center', va='center', transform=ax.transAxes, fontsize=9, color='#555')

plt.tight_layout()
plt.show()

---
## 7. Additional Example: Sampling Quality via the Soup Analogy

> *"It doesn't matter how big your ladle is if the soup is not well-stirred."*

In [ ]:
# --- Sample size vs. bias comparison ---
np.random.seed(1)

# Population: true mean = 50
true_mean   = 50
N_pop       = 100_000
pop_data    = np.random.normal(true_mean, 10, N_pop)

# Biased population: only high values are accessible (convenience sampling)
biased_pop  = pop_data[pop_data > 55]

sizes = [10, 50, 200, 1000, 5000, 10000]

random_means = []
biased_means = []

for n in sizes:
    random_means.append(np.random.choice(pop_data, n).mean())
    biased_means.append(np.random.choice(biased_pop, min(n, len(biased_pop))).mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(sizes, random_means, 'o-', color='#1565C0', linewidth=2,
             markersize=8, label='Random sampling (unbiased)')
ax.semilogx(sizes, biased_means, 's--', color='#E53935', linewidth=2,
             markersize=8, label='Biased sampling (convenience)')
ax.axhline(true_mean, color='#2E7D32', linewidth=1.8,
            linestyle=':', label=f'True mean = {true_mean}')

ax.set_xlabel('Sample Size (log scale)', fontsize=11)
ax.set_ylabel('Sample Mean', fontsize=11)
ax.set_title('Large Biased Sample vs. Small Unbiased Sample',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Observation: As the random sample grows larger, it converges to the true value.')
print('             No matter how large the biased sample becomes, it never converges to the true value!')

---
## Summary

| Concept | Definition | Example |
|--------|-------|-------|
| **Data Matrix** | Rows = observations, Columns = variables | Survey table |
| **Numerical variable** | Continuous or discrete | sleep hours, country count |
| **Categorical variable** | Nominal or ordinal | gender, bedtime |
| **Explanatory variable** | The predictor side | Study hours → GPA |
| **Response variable** | The outcome side | GPA |
| **Observational study** | Data collected without intervention | Survey |
| **Experiment** | Includes random assignment | RCT |
| **Simple RS** | Completely random | Lottery |
| **Stratified sampling** | Proportional from each group | Gender stratum |
| **Cluster sampling** | Entire group selected | Classroom sampling |
| **Sampling bias** | Representation problem | Literary Digest |
| **Association ≠ Causation** | Confounding variable | Shoe size – reading |

---
